# Shared PyTorch DataLoader

**Everyone imports from this notebook** so models see exactly the same
inputs. Configure these four things:

| Parameter | What it controls |
|---|---|
| `features` | Which columns the model sees (see table below) |
| `input_len` | How many past steps the model sees |
| `target_len` | How many future steps to predict |
| `stride` | How far the sliding window advances between samples |

**Inputs** (produced by the earlier notebooks):

- `processed/ais_trajectories_{TIMESTEP}min.parquet` ← `generate_trajectories.ipynb`
- `processed/splits.parquet`                         ← `make_splits.ipynb`
- `processed/feature_stats.csv`                      ← `make_splits.ipynb`

**Output shape per batch**:
```
x: (batch, input_len,  n_features)    # normalized
y: (batch, target_len, n_features)
```
where `n_features` depends on the feature list you pick (angular features
like `cog` expand into 2 columns).

## 1. Imports and configuration

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset, DataLoader

OUT_DIR = "processed"

# ╔══════════════════════════════════════════════════════════════════════╗
# ║  >>> Match the timestep of the parquet file you're using <<<         ║
# ╚══════════════════════════════════════════════════════════════════════╝
TIMESTEP_MIN = 3
REGION = "West Coast"
INTERP_FILE  = f"{OUT_DIR}/ais_trajectories_{REGION}_{TIMESTEP_MIN}min.parquet"
SPLITS_FILE  = f"{OUT_DIR}/splits_{REGION}_{TIMESTEP_MIN}min.parquet"
STATS_FILE   = f"{OUT_DIR}/feature_stats_{REGION}_{TIMESTEP_MIN}min.csv"

## 2. Feature registry

Available features and how they're encoded before going into the model.
The columns come from `ais_trajectories_{TIMESTEP}min.parquet` (19 columns).

| Key | Parquet column | Encoding | Output dims |
|---|---|---|---|
| `lat` | `latitude`  | z-score normalize | 1 |
| `lon` | `longitude` | z-score normalize | 1 |
| `sog` | `sog`       | z-score normalize | 1 |
| `cog` | `cog`       | angular → cos, sin | **2** (`cog_cos`, `cog_sin`) |
| `heading` | `heading` | angular → cos, sin | **2** |
| `length` | `length` | z-score normalize | 1 |
| `width`  | `width`  | z-score normalize | 1 |
| `draft`  | `draft`  | z-score normalize | 1 |

- **Angular** features (`cog`, `heading`) are split into cosine + sine to
  handle the 0°/360° wraparound. They're already in [−1, 1], no normalize.
- **z-score** means `(x - mean) / std` using train-only stats from
  `feature_stats.csv`. NaNs become 0 after normalization (≈ feature mean).
- Categorical columns (vessel_type, transceiver, status) are NOT exposed
  here — add them if your group decides they help. Most AIS prediction
  papers use only position / speed / course.

**Default feature set**: `["lat", "lon", "sog", "cog"]` → 5 model columns
(matches what the reference papers use).

In [ ]:
FEATURE_REGISTRY = {
    "lat":     {"col": "latitude",  "encoding": "normalize"},
    "lon":     {"col": "longitude", "encoding": "normalize"},
    "sog":     {"col": "sog",       "encoding": "normalize"},
    "cog":     {"col": "cog",       "encoding": "angular"},
    "heading": {"col": "heading",   "encoding": "angular"},
    "length":  {"col": "length",    "encoding": "normalize"},
    "width":   {"col": "width",     "encoding": "normalize"},
    "draft":   {"col": "draft",     "encoding": "normalize"},
}

DEFAULT_FEATURES = ["lat", "lon", "sog", "cog", "heading"]   # → 7 columns

# Position encoding for lat/lon. Two options:
#
#   "relative" (default, recommended for LSTM/Transformer):
#       Each window's lat/lon are stored as DISPLACEMENT IN KILOMETERS from
#       the last input timestep, using a local equirectangular projection
#       around the reference latitude:
#           Δlat_km = (lat_deg - ref_lat_deg) * 111.320
#           Δlon_km = (lon_deg - ref_lon_deg) * 111.320 * cos(ref_lat_rad)
#       The model becomes both location-invariant AND scale-consistent: a
#       1-km step north and a 1-km step east are the same magnitude in
#       feature space, regardless of latitude. Loss in km is directly
#       comparable to the haversine evaluation metric (within <1% over
#       <100 km windows).
#       Predictions are also in km space; recover absolute lat/lon with
#       `dataset.recover_absolute(i, delta_km)` (or, for a specific window,
#       `reference_latlon(i)` returns the anchor in degrees and you can
#       invert the projection yourself).
#
#   "absolute":
#       Lat/lon are z-scored against the global train mean/std, like all
#       other normalized features. The model has to memorize the
#       coastline geography to predict positions. Use only if you have a
#       specific reason — relative is strictly better for trajectory
#       forecasting under almost all settings.
DEFAULT_POSITION_ENCODING = "relative"

# Some features weren't included in feature_stats.csv (only lat/lon/sog/cog/heading
# were). We can either re-read the parquet to compute stats for extra columns,
# or require the user to only use features that have stats. For group simplicity,
# we compute stats on the fly for any numeric feature not already in the file.

POSITION_FEATURES = {"lat", "lon"}   # which feature keys are positions

# Equirectangular projection constant. 1° of latitude ≈ 111.320 km on a sphere
# of mean Earth radius. 1° of longitude ≈ 111.320 × cos(lat) km — the cosine
# factor is per-window using the reference (last-input-timestep) latitude.
KM_PER_DEG_LAT = 111.320

## 3. The `AISDataset` class

One instance = one split (train / val / test). Responsibilities:

1. Load the parquet, filter to the requested split via `splits.parquet`.
2. Sort rows so each trip is contiguous and time-ordered.
3. Build a `(N, n_features)` float32 array with the selected features
   (normalized / angular-encoded as appropriate).
4. Precompute `(start, end)` index pairs for every valid sliding window.
   Windows never cross trip boundaries.
5. `__getitem__(i)` slices window `i` — cheap, no data copied until needed.

In [ ]:
class AISDataset(Dataset):
    def __init__(self, split, features=None,
                 input_len=20, target_len=20, stride=5,
                 position_encoding=DEFAULT_POSITION_ENCODING):
        """
        Args:
            split:              "train", "val", or "test"
            features:           list of feature keys from FEATURE_REGISTRY.
                                Default = ["lat","lon","sog","cog","heading"]
                                → 7 model columns (cog, heading expand to cos/sin).
            input_len:          # past steps per sample (default 20 = 1 hr).
            target_len:         # future steps per sample.
            stride:             sliding step between consecutive samples.
                                stride=1 = max overlap; stride=target_len = no overlap.
                                Default 5 = 15-min spacing (good train/eval middle
                                ground; cuts redundant samples ~5x vs stride=1).
            position_encoding:  "relative" (default) or "absolute".
                                See module-level docstring near DEFAULT_POSITION_ENCODING.
        """
        assert split in ("train", "val", "test")
        assert position_encoding in ("relative", "absolute")
        features = features or DEFAULT_FEATURES
        for f in features:
            if f not in FEATURE_REGISTRY:
                raise ValueError(
                    f"Unknown feature '{f}'. Available: {list(FEATURE_REGISTRY)}"
                )

        self.split = split
        self.features = features
        self.input_len = input_len
        self.target_len = target_len
        self.stride = stride
        self.window = input_len + target_len
        self.position_encoding = position_encoding

        # ── Load + filter data ──
        df = pd.read_parquet(INTERP_FILE)
        splits = pd.read_parquet(SPLITS_FILE)
        keep_mmsi = splits.loc[splits["split"] == split, "mmsi"].to_numpy()
        df = df[df["mmsi"].isin(keep_mmsi)].copy()
        df = df.sort_values(["trip_id", "base_date_time"]).reset_index(drop=True)

        # ── Load / compute normalization stats ──
        self.mean, self.std = self._load_stats(features, df)

        # ── Build the feature matrix and the model-feature column names ──
        self.feature_array, self.output_names, self.relative_col_idx = (
            self._build_features(df, features)
        )
        self.n_features = self.feature_array.shape[1]

        # ── Precompute sliding-window indices ──
        self.windows = self._build_windows(df)

        # ── Cache reference (lat, lon) per window (last input timestep, raw deg).
        #    Used by reference_latlon(i) — only meaningful when lat/lon are in
        #    the feature set under "relative" mode, but we always populate so
        #    the helper works regardless.
        if "lat" in features and "lon" in features:
            lat_raw = df["latitude"].to_numpy(dtype=np.float32)
            lon_raw = df["longitude"].to_numpy(dtype=np.float32)
            ref_offsets = self.windows[:, 0] + self.input_len - 1
            self._ref_lat = lat_raw[ref_offsets]
            self._ref_lon = lon_raw[ref_offsets]
            # cos(ref_lat) per window for equirectangular Δlon→km conversion.
            self._cos_ref_lat = np.cos(np.deg2rad(self._ref_lat)).astype(np.float32)
        else:
            self._ref_lat = self._ref_lon = self._cos_ref_lat = None

        # ── Locate the lat/lon columns in the feature matrix (if relative).
        #    Used by __getitem__ to apply the per-window km conversion.
        self._lat_rel_col = None
        self._lon_rel_col = None
        for j, name in enumerate(self.output_names):
            if name == "lat_rel_km":
                self._lat_rel_col = j
            elif name == "lon_rel_km":
                self._lon_rel_col = j
        # Hard requirement for relative-km encoding: if lon is relative, lat
        # must also be present (we need cos(ref_lat) to convert Δlon→km).
        if self._lon_rel_col is not None and self._cos_ref_lat is None:
            raise ValueError(
                "position_encoding='relative' with 'lon' in features requires "
                "'lat' to also be in features (needed to compute cos(ref_lat) "
                "for the Δlon→km equirectangular conversion)."
            )

        print(f"[{split}] {len(df):,} rows, {df['trip_id'].nunique():,} trips, "
              f"{len(self.windows):,} windows   "
              f"features={self.output_names} (n={self.n_features})  "
              f"position_encoding={self.position_encoding}")

    # ------------------------------------------------------------------
    def _load_stats(self, features, df):
        """Load mean/std from feature_stats.csv; fall back to train-set stats."""
        try:
            stats = pd.read_csv(STATS_FILE, index_col=0)
            mean = stats["mean"].to_dict()
            std  = stats["std"].to_dict()
        except FileNotFoundError:
            mean, std = {}, {}

        # For any requested feature whose stats aren't in the file, compute
        # them now from the current (train/val/test) split. WARNING: val/test
        # should not drive stats — call this only if you regenerated stats.
        for f in features:
            cfg = FEATURE_REGISTRY[f]
            if cfg["encoding"] != "normalize":
                continue
            col = cfg["col"]
            if col not in mean:
                print(f"  WARNING: '{col}' not in feature_stats.csv — "
                      f"using current split stats (may leak if split != train)")
                mean[col] = df[col].mean()
                std[col]  = df[col].std()
        return mean, std

    def _build_features(self, df, features):
        """
        Build the (N, n_out) float32 feature matrix.

        Returns:
            arr:         the feature matrix
            names:       column labels (one per output column)
            rel_idx:     output column indices that are "relative position" —
                         these need per-window subtraction in __getitem__.
        """
        cols, names, rel_idx = [], [], []
        for f in features:
            cfg = FEATURE_REGISTRY[f]
            col = cfg["col"]
            vals = df[col].to_numpy(dtype=np.float64)

            if cfg["encoding"] == "normalize":
                if (f in POSITION_FEATURES
                        and self.position_encoding == "relative"):
                    # Store raw degrees; per-window subtraction AND degrees→km
                    # conversion happen in __getitem__ (the cosine factor for
                    # Δlon depends on the per-window reference latitude).
                    cols.append(vals)
                    names.append(f"{f}_rel_km")
                    rel_idx.append(len(cols) - 1)
                else:
                    mu, sd = self.mean[col], self.std[col]
                    sd = sd if sd > 1e-9 else 1.0   # guard against constant cols
                    cols.append((vals - mu) / sd)
                    names.append(f"{f}_norm")

            elif cfg["encoding"] == "angular":
                rad = np.deg2rad(vals)
                cols.append(np.cos(rad))
                cols.append(np.sin(rad))
                names.append(f"{f}_cos")
                names.append(f"{f}_sin")

            else:
                raise ValueError(f"Unknown encoding: {cfg['encoding']}")

        arr = np.stack(cols, axis=1).astype(np.float32)
        # NaN → 0 (mean for z-scored features; neutral for angular and relative)
        arr = np.nan_to_num(arr, nan=0.0)
        return arr, names, rel_idx

    def _build_windows(self, df):
        """Generate (start, end) index pairs — windows never cross trip boundaries."""
        windows = []
        for _, idxs in df.groupby("trip_id").indices.items():
            L = len(idxs)
            if L < self.window:
                continue
            start_global = idxs[0]
            max_start = L - self.window
            for s in range(0, max_start + 1, self.stride):
                gs = start_global + s
                windows.append((gs, gs + self.window))
        return np.asarray(windows, dtype=np.int64)

    # ------------------------------------------------------------------
    def __len__(self):
        return len(self.windows)

    def __getitem__(self, i):
        start, end = self.windows[i]
        seq = self.feature_array[start:end]        # (window, n_features)

        if self.relative_col_idx:
            # Subtract reference (last input timestep) from relative-position
            # columns to get Δdegrees, then convert Δdegrees→Δkm using a
            # local equirectangular projection at the reference latitude.
            # We copy first so the underlying feature_array isn't mutated
            # across calls.
            seq = seq.copy()
            ref = seq[self.input_len - 1, self.relative_col_idx].copy()
            seq[:, self.relative_col_idx] -= ref

            if self._lat_rel_col is not None:
                seq[:, self._lat_rel_col] *= KM_PER_DEG_LAT
            if self._lon_rel_col is not None:
                seq[:, self._lon_rel_col] *= KM_PER_DEG_LAT * self._cos_ref_lat[i]

        x = seq[: self.input_len]                  # (input_len,  n_features)
        y = seq[self.input_len :]                  # (target_len, n_features)
        return torch.from_numpy(x), torch.from_numpy(y)

    # ------------------------------------------------------------------
    def reference_latlon(self, i):
        """Return (ref_lat, ref_lon) in absolute degrees for window `i`.

        This is the position at the last input timestep — i.e., what the
        model is predicting *from*. Under "relative" encoding, predictions
        are Δkm from this anchor; use `recover_absolute(i, delta_km)` to
        invert the projection.

        Only available when both `lat` and `lon` are in the feature list.
        Returns None if not.
        """
        if self._ref_lat is None:
            return None
        return float(self._ref_lat[i]), float(self._ref_lon[i])

    def recover_absolute(self, i, delta_km):
        """Convert a Δkm prediction for window `i` back to absolute (lat, lon) deg.

        Args:
            i:         window index
            delta_km:  array-like of shape (..., 2) with last dim = (Δlat_km, Δlon_km),
                       i.e. the model's prediction in the same space as `y`.

        Returns:
            (abs_lat_deg, abs_lon_deg) — each shape (...,) — absolute degrees.

        Inverts the equirectangular projection used in __getitem__:
            Δlat_deg = Δlat_km / 111.320
            Δlon_deg = Δlon_km / (111.320 * cos(ref_lat_rad))
        and adds the reference. Requires both `lat` and `lon` in features.
        """
        if self._ref_lat is None:
            raise ValueError(
                "recover_absolute requires both 'lat' and 'lon' in features."
            )
        delta_km = np.asarray(delta_km)
        ref_lat = float(self._ref_lat[i])
        ref_lon = float(self._ref_lon[i])
        cos_ref = float(self._cos_ref_lat[i])
        abs_lat = ref_lat + delta_km[..., 0] / KM_PER_DEG_LAT
        abs_lon = ref_lon + delta_km[..., 1] / (KM_PER_DEG_LAT * cos_ref)
        return abs_lat, abs_lon

## 4. Convenience helper: `get_dataloaders()`

Returns `(train, val, test)` DataLoaders — all use the same feature list,
input_len, target_len, and stride, so results are apples-to-apples.

Train-set rebalancing (dropping moored / drifting trips) happens upstream
in `make_splits.ipynb`, so this loader doesn't need to do anything special:
val and test still see the real distribution.

In [ ]:
def get_dataloaders(features=None,
                    input_len=20, target_len=20, stride=5,
                    batch_size=64, num_workers=0,
                    position_encoding=DEFAULT_POSITION_ENCODING):
    train_ds = AISDataset("train", features, input_len, target_len, stride,
                          position_encoding)
    val_ds   = AISDataset("val",   features, input_len, target_len, stride,
                          position_encoding)
    test_ds  = AISDataset("test",  features, input_len, target_len, stride,
                          position_encoding)

    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True,
                              num_workers=num_workers, drop_last=True)
    val_loader   = DataLoader(val_ds,   batch_size=batch_size, shuffle=False,
                              num_workers=num_workers)
    test_loader  = DataLoader(test_ds,  batch_size=batch_size, shuffle=False,
                              num_workers=num_workers)
    return train_loader, val_loader, test_loader

## 5. Demo — default settings (recommended)

Uses everything the dataloader thinks is best for LSTM / Transformer:

- features `["lat", "lon", "sog", "cog", "heading"]` → 7 model columns
- input_len=20, target_len=20 (1 hour in, 1 hour out)
- stride=5 (15-min spacing between training windows)
- position_encoding="relative" (lat/lon are Δkm from the last input step)

The demo only runs when this file is executed directly (or as a notebook).
`from dataloader import get_dataloaders` does NOT trigger the demo —
importing the module pay only the import cost, not the
split-loading cost.

In [ ]:
if __name__ == "__main__":
    train_loader, val_loader, test_loader = get_dataloaders(batch_size=64)

    x, y = next(iter(train_loader))
    print(f"\nOne training batch:")
    print(f"  x.shape = {tuple(x.shape)}   dtype={x.dtype}")
    print(f"  y.shape = {tuple(y.shape)}   dtype={y.dtype}")

    names = train_loader.dataset.output_names
    print(f"\nFeature stats in this batch:")
    for i, name in enumerate(names):
        print(f"  {name:12s}  mean={x[..., i].mean().item():+.3f}  "
              f"std={x[..., i].std().item():.3f}")

    # Sanity: under "relative" encoding, the lat/lon column at the LAST input
    # step should be exactly 0 (it IS the reference). Quick check:
    lat_idx = names.index("lat_rel_km")
    lon_idx = names.index("lon_rel_km")
    last_input = x[:, -1, [lat_idx, lon_idx]]
    print(f"\nReference check (should be ~0): "
          f"max |lat_rel_km[t=last_input]| = {last_input[:, 0].abs().max():.2e}, "
          f"max |lon_rel_km[t=last_input]| = {last_input[:, 1].abs().max():.2e}")

    # Magnitude check: Δkm over a 1-hour future window with sog up to ~30 kn
    # ≈ ~55 km max displacement. Print actual range to confirm units.
    y_lat_km = y[..., lat_idx]
    y_lon_km = y[..., lon_idx]
    print(f"\nFuture-window Δkm magnitude (sample of {y.shape[0]} windows):")
    print(f"  lat_rel_km:  min={y_lat_km.min():+.2f}  max={y_lat_km.max():+.2f}  "
          f"mean|·|={y_lat_km.abs().mean():.2f}  km")
    print(f"  lon_rel_km:  min={y_lon_km.min():+.2f}  max={y_lon_km.max():+.2f}  "
          f"mean|·|={y_lon_km.abs().mean():.2f}  km")

[train] 57,614,788 rows, 180,280 trips, 10,306,417 windows   features=['lat_rel_km', 'lon_rel_km', 'sog_norm', 'cog_cos', 'cog_sin', 'heading_cos', 'heading_sin'] (n=7)  position_encoding=relative


[val] 34,915,380 rows, 76,659 trips, 6,457,737 windows   features=['lat_rel_km', 'lon_rel_km', 'sog_norm', 'cog_cos', 'cog_sin', 'heading_cos', 'heading_sin'] (n=7)  position_encoding=relative


[test] 35,937,927 rows, 74,149 trips, 6,678,696 windows   features=['lat_rel_km', 'lon_rel_km', 'sog_norm', 'cog_cos', 'cog_sin', 'heading_cos', 'heading_sin'] (n=7)  position_encoding=relative

One training batch:
  x.shape = (64, 20, 7)   dtype=torch.float32
  y.shape = (64, 20, 7)   dtype=torch.float32

Feature stats in this batch:
  lat_rel_km    mean=+0.338  std=4.599
  lon_rel_km    mean=+1.229  std=4.831
  sog_norm      mean=+0.016  std=1.118
  cog_cos       mean=+0.058  std=0.738
  cog_sin       mean=-0.030  std=0.598
  heading_cos   mean=+0.031  std=0.609
  heading_sin   mean=+0.025  std=0.534



Reference check (should be ~0): max |lat_rel_km[t=last_input]| = 0.00e+00, max |lon_rel_km[t=last_input]| = 0.00e+00

Future-window Δkm magnitude (sample of 64 windows):
  lat_rel_km:  min=-29.41  max=+116.22  mean|·|=1.72  km
  lon_rel_km:  min=-145.59  max=+40.61  mean|·|=2.81  km


## 6. Recovering absolute lat/lon from a Δkm prediction

The model outputs `y_hat` in the same Δkm space as `y`. To plot on a map
(or compute haversine distance to a known waypoint), invert the
equirectangular projection — `recover_absolute` does the right thing:

```python
i = some_window_index_in_the_eval_loader
# y_hat[i] has shape (target_len, n_features); pick the lat/lon channels:
delta_km = y_hat[i][:, [lat_idx, lon_idx]]               # (target_len, 2)
abs_lat, abs_lon = test_loader.dataset.recover_absolute(i, delta_km)
```

Or, manually:

```python
ref_lat, ref_lon = test_loader.dataset.reference_latlon(i)   # absolute degrees
cos_ref = np.cos(np.deg2rad(ref_lat))
abs_lat = ref_lat + y_hat[i, :, lat_idx] / 111.320
abs_lon = ref_lon + y_hat[i, :, lon_idx] / (111.320 * cos_ref)
```

For training loss / MAE you don't need to convert at all — Δkm error is
(within <1%) the same as the haversine eval metric you actually care about,
which is the whole point of the km encoding.

In [ ]:
if __name__ == "__main__":
    ref = test_loader.dataset.reference_latlon(0)
    print(f"reference_latlon(window 0) = {ref}  "
          f"(absolute lat/lon at the last input timestep)")

    # Round-trip sanity check: take y[0] (the true future Δkm), invert it,
    # and confirm we recover absolute lat/lon trajectories with reasonable
    # magnitudes (within the West Coast bounding box).
    y0 = y[0][:, [lat_idx, lon_idx]].numpy()         # (target_len, 2) Δkm
    abs_lat, abs_lon = test_loader.dataset.recover_absolute(0, y0)
    print(f"recover_absolute(0, y[0]) → lat ∈ [{abs_lat.min():.4f}, {abs_lat.max():.4f}], "
          f"lon ∈ [{abs_lon.min():.4f}, {abs_lon.max():.4f}]")

reference_latlon(window 0) = (43.862449645996094, -124.79525756835938)  (absolute lat/lon at the last input timestep)
recover_absolute(0, y[0]) → lat ∈ [43.8625, 43.8639], lon ∈ [-124.7963, -124.7953]


## 7. How your model code uses this

```python
from dataloader import get_dataloaders

train_loader, val_loader, test_loader = get_dataloaders(batch_size=128)
n_features = train_loader.dataset.n_features

model = LSTMForecaster(n_features=n_features, hidden=128)
for x, y in train_loader:
    y_hat = model(x)                 # (B, target_len, n_features)
    loss = F.mse_loss(y_hat, y)      # loss is in km² ≈ haversine²
```

**If you specifically want absolute z-scored coords** (not recommended
for new models, but provided for compatibility):

```python
train_loader, val_loader, test_loader = get_dataloaders(
    position_encoding="absolute",
)
# Then to denormalize lat/lon predictions to degrees:
stats = pd.read_csv("processed/feature_stats.csv", index_col=0)
lat_deg = y_hat[..., 0].numpy() * stats.loc["latitude",  "std"] + stats.loc["latitude",  "mean"]
lon_deg = y_hat[..., 1].numpy() * stats.loc["longitude", "std"] + stats.loc["longitude", "mean"]
```